# Organzing M2OR
## https://academic.oup.com/nar/article/52/D1/D1370/7327067
## https://m2or.chemsensim.fr/

In [1]:
from pathlib import Path
import pandas as pd

PAIRS_CSV          = Path("pairs.csv")
COMPOUNDS_CSV      = Path("compounds.csv")
RECEPTORS_CSV      = Path("receptors.csv")
MAIN_COMPOUNDS_CSV = Path("main_compounds.csv")
BLAST_CSV          = Path("blast.csv")
OUT_CSV            = Path("M2OR_pair_processed.csv")

def read_semicolon_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=";", dtype=str, engine="python")
    return df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

def main():
    pairs          = read_semicolon_csv(PAIRS_CSV)
    compounds      = read_semicolon_csv(COMPOUNDS_CSV)
    receptors      = read_semicolon_csv(RECEPTORS_CSV)
    main_compounds = read_semicolon_csv(MAIN_COMPOUNDS_CSV)
    blast          = read_semicolon_csv(BLAST_CSV)

    for df, cols in [
        (pairs, ["main_compounds_id", "main_receptors_id", "mutated_sequence"]),
        (compounds, ["main_compounds_id"]),
        (receptors, ["main_receptors_id", "gene_id"]),
        (main_compounds, ["main_compounds_id"]),
        (blast, ["mutated_sequence", "blast_gene_name", "species"]),
    ]:
        for c in cols:
            if c in df.columns:
                df[c] = df[c].astype(str)

    mc_map = (main_compounds[["main_compounds_id", "mixture"]]
              .dropna(subset=["main_compounds_id"])
              .drop_duplicates(subset=["main_compounds_id"], keep="first"))

    comp_map = (compounds[["main_compounds_id", "name"]]
                .dropna(subset=["main_compounds_id"])
                .drop_duplicates(subset=["main_compounds_id"], keep="first"))

    rcp_map = (receptors[["main_receptors_id", "gene_id"]]
               .dropna(subset=["main_receptors_id"])
               .drop_duplicates(subset=["main_receptors_id"], keep="first"))

    blast_seq_map = (blast[["mutated_sequence", "species"]]
                     .dropna(subset=["mutated_sequence"])
                     .drop_duplicates(subset=["mutated_sequence"], keep="first"))

    blast_gene_map = (blast[["blast_gene_name", "species"]]
                      .dropna(subset=["blast_gene_name"])
                      .drop_duplicates(subset=["blast_gene_name"], keep="first")
                      .rename(columns={"blast_gene_name": "gene_id"}))

    enriched = pairs.merge(mc_map, on="main_compounds_id", how="left")
    enriched = enriched.merge(comp_map, on="main_compounds_id", how="left")
    enriched = enriched.merge(rcp_map, on="main_receptors_id", how="left")

    enriched = enriched.merge(blast_seq_map, on="mutated_sequence", how="left")

    if "species" in enriched.columns:
        mask_missing_species = enriched["species"].isna() | (enriched["species"] == "")
        if mask_missing_species.any():
            fallback = enriched.loc[mask_missing_species, ["gene_id"]].merge(
                blast_gene_map, on="gene_id", how="left"
            )[["species"]]
            enriched.loc[mask_missing_species, "species"] = fallback["species"].values

    original_cols = list(pairs.columns)
    new_cols = [c for c in ["mixture", "name", "gene_id", "species"] if c in enriched.columns]
    enriched = enriched[original_cols + new_cols]

    enriched[new_cols] = enriched[new_cols].fillna("")

    enriched.to_csv(OUT_CSV, sep=";", index=False)
    print(f"Saved: {OUT_CSV.resolve()}")

if __name__ == "__main__":
    main()


/var/folders/ts/n9fbfxkj091_fpcsbsgj4zcxb6n5sd/T/ipykernel_48255/2118756455.py:13: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/var/folders/ts/n9fbfxkj091_fpcsbsgj4zcxb6n5sd/T/ipykernel_48255/2118756455.py:13: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/var/folders/ts/n9fbfxkj091_fpcsbsgj4zcxb6n5sd/T/ipykernel_48255/2118756455.py:13: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/var/folders/ts/n9fbfxkj091_fpcsbsgj4zcxb6n5sd/T/ipykernel_48255/2118756455.py:13: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/var/folders/ts/n9fbfxkj091_fpcsbsgj4zcxb6n5sd/T/ipykern

Saved: /Users/vs479/Desktop/Desktop/HNL/OlfactoryInteligence_Benchmark/6-Hard_1/Data/M2OR_pair_processed.csv
